Check that $\nabla\cdot B=0$, $B\cdot\nabla\psi=0$, and $(\nabla\times B)\times
B = \nabla p$ for the sheared-iota family directly from the explicit formulas
for $B, \psi$, and $p$ using automatic differentiation for the derivatives.

In [44]:
import jax
# Tell jax to use double precision
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
from jax import jacfwd, vmap

In [45]:
eps = 0.1
S = 0.18
lambd = 0.34
pa = 0.41

I = 1.0j

# Altering any of these "mutation" values to be different than 1 should cause the
# assertions at the end to fail.
mutation_1 = 1
mutation_2 = 1

def compute(x, y, z):
    K = (x - I * y) * jnp.sqrt(1 + eps / (x - mutation_1 * I * y)**2)
    Phi = (x + I * y) * K + jnp.pi / 2 - S
    Bx = -(1 / 2) * jnp.imag(jnp.exp(I * lambd * z) * jnp.sin(Phi) / K)
    By =  (1 / 2) * jnp.real(jnp.exp(I * lambd * z) * jnp.sin(Phi) / K)
    Bz = -(1 / lambd) * jnp.real(jnp.exp(I * lambd * z) * jnp.cos(Phi))
    psi = (1/2) * (jnp.sin(lambd * z)**2 + jnp.real(jnp.exp(I * lambd * z) * jnp.cos(Phi))**2)
    p = pa - mutation_2 * (1 / lambd)**2 * psi
    return Bx, By, Bz, psi, p

n_points = 100
R = (np.random.rand(n_points) - 0.5) * 0.1 + 1
zeta = (np.random.rand(n_points) - 0.5) * 2 * np.pi
z = (np.random.rand(n_points) - 0.5) * 0.1
x = R * np.cos(zeta)
y = R * np.sin(zeta)

Bx, By, Bz, psi, p = compute(x, y, z)

# jacfwd+vmap gives d(output)/d(x,y,z) at each of the n_points, since compute is elementwise
jac_fn = vmap(jacfwd(compute, argnums=(0, 1, 2)))
dBx, dBy, dBz, dpsi, dp = jac_fn(x, y, z)

dBx_dx, dBx_dy, dBx_dz = dBx
dBy_dx, dBy_dy, dBy_dz = dBy
dBz_dx, dBz_dy, dBz_dz = dBz
dpsi_dx, dpsi_dy, dpsi_dz = dpsi
dp_dx, dp_dy, dp_dz = dp      
assert dBx_dx.shape == (n_points,)
assert dp_dx.shape == (n_points,)

Jx = dBz_dy - dBy_dz
Jy = dBx_dz - dBz_dx
Jz = dBy_dx - dBx_dy

J_cross_B_x = Jy * Bz - Jz * By
J_cross_B_y = Jz * Bx - Jx * Bz
J_cross_B_z = Jx * By - Jy * Bx

Check that $\nabla\cdot B = 0$:

In [46]:
div_B = dBx_dx + dBy_dy + dBz_dz
max_abs_div_B = np.max(np.abs(div_B))
print(f"Maximum |div B|: {max_abs_div_B}")
np.testing.assert_allclose(div_B, 0.0, rtol=0, atol=1e-14)
assert max_abs_div_B > 0  # Should be comparable to roundoff
assert max_abs_div_B < 1e-15

Maximum |div B|: 1.8735013540549517e-16


Check that $B \cdot\nabla\psi = 0$:

In [47]:
B_dot_grad_psi = Bx * dpsi_dx + By * dpsi_dy + Bz * dpsi_dz
max_abs_B_dot_grad_psi = np.max(np.abs(B_dot_grad_psi))
print(f"Maximum |B dot grad psi|: {max_abs_B_dot_grad_psi}")
np.testing.assert_allclose(max_abs_B_dot_grad_psi, 0.0, rtol=0, atol=1e-14)
assert max_abs_B_dot_grad_psi > 0  # Should be comparable to roundoff
assert max_abs_B_dot_grad_psi < 1e-15

Maximum |B dot grad psi|: 6.071532165918825e-17


Check that $J \times B = \nabla p$:

In [49]:
force_residual = np.array(
    [
        J_cross_B_x - dp_dx,
        J_cross_B_y - dp_dy,
        J_cross_B_z - dp_dz,
    ]
)
max_abs_force_residual = np.max(np.abs(force_residual))
print(f"Maximum |force residual|: {max_abs_force_residual}")
np.testing.assert_allclose(max_abs_force_residual, 0.0, rtol=0, atol=1e-14)
assert max_abs_force_residual > 0  # Should be comparable to roundoff
assert max_abs_force_residual < 1e-14

Maximum |force residual|: 3.552713678800501e-15
